# Real-world Data Project: Retail Sales Analysis
## Sales Performance Analysis and Prediction

**Submitted by:** ____________________  
**Course:** ____________________  
**Date:** ____________________

### Objective
Analyze retail sales data to find business trends and build a model to predict sales from product, region, price, discount, and customer information.


## 1. Import Libraries
Install packages once if required:

`pip install pandas numpy matplotlib seaborn scikit-learn jupyter`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='deep')


## 2. Load and Inspect the Dataset


In [ ]:
df = pd.read_csv('dataset/retail_sales_data.csv')
df.head()

print('Dataset shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nStatistical summary:')
print(df.describe())


## 3. Clean the Data
Missing unit prices and customer ratings are replaced with their median values. Duplicate orders are removed.


In [ ]:
clean = df.drop_duplicates(subset='Order_ID').copy()
clean['Order_Date'] = pd.to_datetime(clean['Order_Date'])
clean['Unit_Price'] = clean['Unit_Price'].fillna(clean['Unit_Price'].median())
clean['Customer_Rating'] = clean['Customer_Rating'].fillna(clean['Customer_Rating'].median())
clean['Month'] = clean['Order_Date'].dt.to_period('M').astype(str)
clean.to_csv('dataset/cleaned_retail_sales_data.csv', index=False)
print('Clean shape:', clean.shape)
print('Remaining missing values:', clean.isnull().sum().sum())


## 4. Exploratory Data Analysis


In [ ]:
monthly_sales = clean.groupby('Month', as_index=False)['Sales'].sum()
category_sales = clean.groupby('Category', as_index=False)['Sales'].sum()
plt.figure(figsize=(11, 5))
sns.lineplot(data=monthly_sales, x='Month', y='Sales', marker='o', color='#2563eb')
plt.xticks(rotation=45)
plt.title('Monthly Sales Trend')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=category_sales, x='Category', y='Sales', color='#7c3aed')
plt.title('Total Sales by Product Category')
plt.xticks(rotation=15)
plt.show()


In [ ]:
region_profit = clean.groupby('Region', as_index=False)['Profit'].sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=region_profit, x='Region', y='Profit', ax=axes[0], color='#10b981')
axes[0].set_title('Total Profit by Region')
sns.scatterplot(data=clean, x='Units_Sold', y='Sales', hue='Category', alpha=0.65, ax=axes[1])
axes[1].set_title('Sales vs Units Sold')
plt.tight_layout()
plt.show()


## 5. Key Business Findings
The charts show which categories and regions generate the most revenue and profit. Monthly trend analysis can identify seasonal peaks, while the scatter plot shows how units sold relates to sales value.


In [ ]:
print('Sales by category:')
print(clean.groupby('Category')['Sales'].sum().sort_values(ascending=False))
print('\nProfit by region:')
print(clean.groupby('Region')['Profit'].sum().sort_values(ascending=False))
print('\nAverage sales by discount:')
print(clean.groupby('Discount')['Sales'].mean().sort_values(ascending=False))


## 6. Prepare Data for Sales Prediction
The target variable is `Sales`. The model uses region, category, units sold, unit price, discount, and customer rating as input features.


In [ ]:
features = ['Region', 'Category', 'Units_Sold', 'Unit_Price', 'Discount', 'Customer_Rating']
X = clean[features]
y = clean['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

numeric_features = ['Units_Sold', 'Unit_Price', 'Discount', 'Customer_Rating']
categorical_features = ['Region', 'Category']
preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])


## 7. Train and Evaluate the Prediction Model
A Random Forest Regressor is used because it can learn non-linear relationships between sales and multiple business factors.


In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42))
])
model.fit(X_train, y_train)
predicted_sales = model.predict(X_test)

mae = mean_absolute_error(y_test, predicted_sales)
rmse = mean_squared_error(y_test, predicted_sales) ** 0.5
r2 = r2_score(y_test, predicted_sales)
print(f'Mean Absolute Error: ${mae:,.2f}')
print(f'Root Mean Squared Error: ${rmse:,.2f}')
print(f'R² Score: {r2:.3f}')


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, predicted_sales, alpha=0.7, color='#2563eb')
limits = [min(y_test.min(), predicted_sales.min()), max(y_test.max(), predicted_sales.max())]
plt.plot(limits, limits, '--', color='red', label='Perfect prediction')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales')
plt.legend()
plt.show()


## 8. Conclusion

This project applied data science skills to a retail business setting. After cleaning the data, sales trends were compared across time, category, and region. The analysis identifies high-performing areas and shows that units sold, unit price, discounts, and product category are useful for predicting sales. The prediction model can support basic sales planning and inventory decisions.

**Limitation:** The included file is a realistic educational sample, used because no business dataset was provided. Replace it with a real retail CSV for a company-specific analysis.
